<a href="https://colab.research.google.com/github/ataulhaque/ML/blob/main/LangGraph_Compliance_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from typing import TypedDict, Annotated, List, Literal
from operator import add

# Install necessary libraries:
# pip install langgraph langchain-core langchain-openai pydantic

from langgraph.graph import StateGraph, END, START
from langchain_core.messages import HumanMessage
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# --- 0. Setup and Definitions ---

# Set your API Key (e.g., from environment variables)
# The user needs to set their OPENAI_API_KEY environment variable.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
try:
    # Initialize the LLM (using a fast model for the check)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
except Exception as e:
    print(f"Warning: Could not initialize ChatOpenAI. Ensure OPENAI_API_KEY is set. Error: {e}")
    llm = None # Set to None if initialization fails

# Define the State of the LangGraph application using TypedDict
class ComplianceState(TypedDict):
    """
    Represents the state of the compliance pipeline.
    """
    audio_file_path: str # The input path to the audio file
    transcript: str      # The text generated by the STT tool
    compliance_result: Literal["COMPLIANT", "NON_COMPLIANT", "ERROR"] # Result of the check
    compliance_report: str # Detailed report from the compliance LLM

# Pydantic schema for structured output from the compliance checker LLM
class ComplianceCheck(BaseModel):
    """Structured output for the compliance checking LLM."""
    result: Literal["COMPLIANT", "NON_COMPLIANT"] = Field(
        description="The final classification of the transcript content."
    )
    reasoning: str = Field(
        description="A detailed, concise reason for the result, highlighting any non-compliant phrases if applicable."
    )

# --- 1. Tool/Node Functions ---

def transcribe_audio(state: ComplianceState) -> ComplianceState:
    """
    Node 1: Converts the audio/video file into a text transcript.
    In a real application, this would call a Speech-to-Text API (e.g., Google, Whisper, AWS).
    """
    audio_path = state['audio_file_path']
    print(f"\n--- STEP 1: TRANSCRIBING AUDIO from {audio_path} ---")

    # Mock STT operation: we check the filename to simulate different outcomes
    if "compliant_interview" in audio_path:
        transcript = "Hello team, this quarter's results are very promising, and we should focus on sustainable, ethical growth in the next fiscal year."
        print("Mock Transcript: Compliant.")
    elif "non_compliant_rant" in audio_path:
        transcript = "I'm extremely frustrated with that supplier! They are totally unreliable and I will blacklist them right away. This is totally unacceptable business practice."
        print("Mock Transcript: Non-Compliant (Frustration/Threats).")
    else:
        # Simulate an error/generic path
        transcript = "The transcribed audio says: Hello."
        print("Mock Transcript: Default/Generic.")

    # Update state with the generated transcript
    return {"transcript": transcript, "compliance_result": "PROCESSING", "compliance_report": ""}

def check_compliance(state: ComplianceState) -> ComplianceState:
    """
    Node 2: Processes the transcript using an LLM (RAG-enhanced) for compliance checking.
    The LLM uses the ComplianceCheck Pydantic schema for reliable structured output.
    """
    if not llm:
        print("\n--- STEP 2: SKIPPING COMPLIANCE CHECK (LLM not initialized) ---")
        return {"compliance_result": "ERROR", "compliance_report": "LLM not configured."}

    transcript = state['transcript']
    print("\n--- STEP 2: CHECKING COMPLIANCE ---")

    # Define the system prompt for the compliance LLM
    # In a real RAG setup, we would retrieve policies and add them to the prompt context.
    # We simulate this by defining the 'knowledge' directly in the system prompt.
    COMPLIANCE_SYSTEM_PROMPT = """
    You are an AI Compliance Auditor. Your task is to analyze the provided transcript
    against standard corporate policy which prohibits:
    1. Hate speech, discrimination, or abusive language.
    2. Explicitly threatening or blacklisting competitors/partners/suppliers.
    3. Disclosure of sensitive financial information without prior authorization.

    Based ONLY on the transcript, classify it as 'COMPLIANT' or 'NON_COMPLIANT'.
    If non-compliant, you MUST provide a specific reason and quote the offending text in your reasoning.
    """

    # Create a runnable chain with structured output
    compliance_chain = (
        ChatPromptTemplate.from_messages([
            ("system", COMPLIANCE_SYSTEM_PROMPT),
            ("human", "Analyze the following transcript: \n\n{transcript}")
        ])
        | llm.with_structured_output(ComplianceCheck)
    )

    # Invoke the chain
    try:
        response: ComplianceCheck = compliance_chain.invoke({"transcript": transcript})

        # Update state with the results from the structured output
        return {
            "compliance_result": response.result,
            "compliance_report": response.reasoning
        }
    except Exception as e:
        print(f"Error during compliance check: {e}")
        return {"compliance_result": "ERROR", "compliance_report": f"LLM Call Failed: {e}"}

def generate_report(state: ComplianceState) -> ComplianceState:
    """
    Node 3a: Final step for compliant content - generate a simple confirmation report.
    """
    print("\n--- STEP 3a: GENERATING COMPLIANCE REPORT (COMPLIANT) ---")
    final_report = (
        f"✅ Compliance Check Complete: COMPLIANT\n"
        f"Reasoning: {state['compliance_report']}\n"
        f"Action Taken: Archiving transcript and report."
    )
    print(final_report)
    return {"compliance_report": final_report}

def flag_content(state: ComplianceState) -> ComplianceState:
    """
    Node 3b: Final step for non-compliant content - flag for human review.
    """
    print("\n--- STEP 3b: FLAGGING CONTENT FOR REVIEW (NON-COMPLIANT) ---")
    final_report = (
        f"🚨 Compliance Check Complete: NON-COMPLIANT\n"
        f"Reasoning: {state['compliance_report']}\n"
        f"Action Taken: Content is flagged for immediate human review and potential deletion."
    )
    print(final_report)
    return {"compliance_report": final_report}

def handle_error(state: ComplianceState) -> ComplianceState:
    """
    Node 3c: Final step for error state.
    """
    print("\n--- STEP 3c: HANDLING ERROR ---")
    final_report = (
        f"❌ Compliance Check Failed: ERROR\n"
        f"Error Details: {state['compliance_report']}\n"
        f"Action Taken: Content moved to a holding queue for manual investigation."
    )
    print(final_report)
    return {"compliance_report": final_report}


# --- 2. Routing/Conditional Edge Function ---

def route_result(state: ComplianceState) -> str:
    """
    Conditional edge function that determines the next step based on the compliance result.
    """
    result = state['compliance_result']
    print(f"\n--- ROUTING: Result is {result} ---")

    if result == "COMPLIANT":
        return "compliant"
    elif result == "NON_COMPLIANT":
        return "non_compliant"
    else: # Handles "ERROR" and any other unexpected state
        return "error"

# --- 3. Build the LangGraph Workflow ---

def create_compliance_graph():
    # 1. Define the graph with the state
    workflow = StateGraph(ComplianceState)

    # 2. Add the nodes (steps)
    workflow.add_node("transcribe", transcribe_audio)
    workflow.add_node("check_compliance", check_compliance)
    workflow.add_node("generate_report", generate_report)
    workflow.add_node("flag_content", flag_content)
    workflow.add_node("handle_error", handle_error)

    # 3. Define the entry point (first node)
    workflow.set_entry_point("transcribe")

    # 4. Define the sequential edge (after transcription, always check compliance)
    workflow.add_edge("transcribe", "check_compliance")

    # 5. Define the conditional edges (the router)
    workflow.add_conditional_edges(
        "check_compliance", # From the compliance check node
        route_result,       # Using the routing function
        {                   # Mapping of return values to the next node
            "compliant": "generate_report",
            "non_compliant": "flag_content",
            "error": "handle_error",
        }
    )

    # 6. Define the end points
    workflow.add_edge("generate_report", END)
    workflow.add_edge("flag_content", END)
    workflow.add_edge("handle_error", END)

    # 7. Compile the graph
    app = workflow.compile()
    return app

# --- 4. Running the Pipeline ---

if __name__ == "__main__":
    app = create_compliance_graph()

    # Define test inputs (simulated file paths)
    test_cases = [
        "audio_files/compliant_interview_q3_2025.mp4",
        "audio_files/non_compliant_rant_supplier_review.mp3",
        "audio_files/generic_file_001.wav"
    ]

    # Run the graph for each test case
    for input_path in test_cases:
        print("="*80)
        print(f"STARTING WORKFLOW FOR: {input_path}")

        # Initial state payload
        initial_state = {"audio_file_path": input_path, "transcript": "", "compliance_result": "ERROR", "compliance_report": ""}

        # Invoke the graph
        final_state = app.invoke(initial_state)

        print("\n" + "="*20 + " FINAL RESULT " + "="*20)
        print(f"Input File: {final_state['audio_file_path']}")
        print(f"Final Status: {final_state['compliance_result']}")
        print(f"Final Action:\n{final_state['compliance_report']}")
        print("="*80 + "\n")

    print("\n")